In [1]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
# Initialize Spark session
spark = (
    SparkSession.builder.appName("Cleaning")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

25/10/13 08:59:14 WARN Utils: Your hostname, KD4SH1-BFMG resolves to a loopback address: 127.0.1.1; using 10.135.51.49 instead (on interface wlan0)
25/10/13 08:59:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/13 08:59:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
minio_client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

bucket_name = "pulse-bucket-1"

In [4]:
objects = minio_client.list_objects(bucket_name, prefix="mapped_", recursive=True)
dataframes = {}
for obj in objects:
    df = spark.read.csv(
        f"s3a://{bucket_name}/{obj.object_name}", header=True, inferSchema=True
    )
    object_name = obj.object_name.replace("mapped_", "").replace(".csv", "")
    dataframes[object_name] = df

25/10/13 08:59:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
for table in dataframes.keys():
    df = dataframes[table]

    for column in df.columns:
        if column.endswith("_id"):
            df = df.withColumn(
                column,
                when(
                    regexp_extract(col(column), r"(\d+)", 1) == "",
                    None,
                ).otherwise(regexp_extract(col(column), r"(\d+)", 1)),
            )
            df = df.withColumn(column, col(column).cast("int"))

    dataframes[table] = df
for table, dataframe in dataframes.items():
    print(f"Table: {table}")
    dataframe.show(3)

Table: addresses
+----------+------------------+--------------+-----------+--------------+
|address_id|              city|state_province|postal_code|       country|
+----------+------------------+--------------+-----------+--------------+
|      5555|North Derrickmouth|      Sevilla*|       2944|      Slovénie|
|      5938|             Husum|         Idaho|    PH0 6RX|        Rwanda|
|      5060|              NULL|          NULL|    00000  |United Kingdom|
+----------+------------------+--------------+-----------+--------------+
only showing top 3 rows

Table: categories
+-----------+----------+---------------+
|category_id|  category|   sub_category|
+-----------+----------+---------------+
|        695|  Colthing|      Wholesale|
|        659|Mirrorless|Limited Edition|
|        571|     Suits|      Wholesale|
+-----------+----------+---------------+
only showing top 3 rows

Table: customer_sessions
+----------+-----------+--------------------+--------------------+-----------+-------

In [6]:
spark.stop()